In [7]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
import pandas as pd
import numpy as np
from pathlib import Path
from shapely.ops import unary_union
from pyproj import Transformer

# Load all US counties, reproject to match raster
counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip")
counties = counties.to_crs("EPSG:5070")

# Keep only Iowa (STATEFP == "19") and dissolve into one boundary for masking
iowa = counties[counties["STATEFP"] == "19"]
iowa_boundary = [unary_union(iowa.geometry)]

pixel_area_ha = 250 * 250 / 10000  # 6.25 ha

# Transformer from raster CRS (EPSG:5070, meters) to lat/lon (EPSG:4326)
to_latlon = Transformer.from_crs("EPSG:5070", "EPSG:4326", always_xy=True)

# Find all surplus tif files
surplus_dir = Path("../../data/Surplus")
tif_files = sorted(surplus_dir.glob("Surplus_N_*.tif"))
print(f"Found {len(tif_files)} files: {[f.name for f in tif_files]}")

grid_df = None        # static lookup: pixel_id -> x, y, lon, lat (built once)
ref_shape = None      # (height, width) of the cropped Iowa grid, used as a consistency check
ref_transform = None
all_years = []

for tif_path in tif_files:
    year = int(tif_path.stem.split("_")[-1])  # extract year from filename
    print(f"Processing {year}...")

    with rasterio.open(tif_path) as src:
        nodata = src.nodata  # use file's nodata if set, else None
        out_image, out_transform = rio_mask(
            src,
            iowa_boundary,
            crop=True,      # crop to Iowa's bounding box, preserving native 250m grid
            nodata=nodata,  # pixels outside the Iowa polygon get set to nodata
            filled=True
        )

    band = out_image[0]  # single-band raster
    height, width = band.shape

    if ref_shape is None:
        # First file: lock in the reference grid and build the coordinate lookup
        # for EVERY cell in the cropped extent (not just cells valid this year),
        # so pixel_ids line up even if nodata footprints shift across years.
        ref_shape, ref_transform = (height, width), out_transform

        rr, cc = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
        rr, cc = rr.ravel(), cc.ravel()
        full_pixel_id = (rr.astype(np.int64) * width + cc).astype(np.int32)

        xs, ys = rasterio.transform.xy(out_transform, rr, cc)
        xs, ys = np.array(xs), np.array(ys)
        lons, lats = to_latlon.transform(xs, ys)

        grid_df = pd.DataFrame({
            "pixel_id": full_pixel_id,
            "x": xs.astype("float32"),
            "y": ys.astype("float32"),
            "lon": np.asarray(lons, dtype="float32"),
            "lat": np.asarray(lats, dtype="float32"),
        })
    else:
        assert (height, width) == ref_shape, (
            f"{tif_path.name} has a different cropped shape than prior years "
            f"({(height, width)} vs {ref_shape}); pixel_ids would not line up."
        )

    # Identify valid (non-nodata) pixels for this year
    if nodata is not None and not np.isnan(nodata):
        valid_mask = band != nodata
    else:
        valid_mask = ~np.isnan(band)

    rows, cols = np.where(valid_mask)
    values = band[rows, cols].astype("float32")
    pixel_id = (rows.astype(np.int64) * width + cols).astype(np.int32)

    df = pd.DataFrame({
        "pixel_id": pixel_id,
        "year": np.int16(year),
        "surplus_kgha": values,
    })
    df["total_kg_N"] = (df["surplus_kgha"] * pixel_area_ha).astype("float32")
    all_years.append(df)

# Combine into a pixel-level panel (coordinates live separately in grid_df)
panel = pd.concat(all_years, ignore_index=True)

grid_df.to_parquet("iowa_grid_lookup.parquet", index=False)
panel.to_parquet("nitrogen_surplus_iowa_grid_panel.parquet", index=False, compression="snappy")

print(f"Grid cells (lookup table): {len(grid_df):,}")
print(f"Panel rows (pixel x year): {len(panel):,}")
print(panel.head())

Found 6 files: ['Surplus_N_2012.tif', 'Surplus_N_2013.tif', 'Surplus_N_2014.tif', 'Surplus_N_2015.tif', 'Surplus_N_2016.tif', 'Surplus_N_2017.tif']
Processing 2012...
Processing 2013...
Processing 2014...
Processing 2015...
Processing 2016...
Processing 2017...
Grid cells (lookup table): 2,996,074
Panel rows (pixel x year): 17,976,444
   pixel_id  year  surplus_kgha  total_kg_N
0         0  2012           0.0         0.0
1         1  2012           0.0         0.0
2         2  2012           0.0         0.0
3         3  2012           0.0         0.0
4         4  2012           0.0         0.0


,pixel_id,year,surplus_kgha,total_kg_N
585876,585876,2017,137.330002,858.312500
437482,437482,2017,92.949997,580.937500
631341,631341,2017,129.009995,806.312439
1603286,1603286,2017,154.149994,963.437439
2572849,2572849,2017,0.000000,0.000000


In [1]:
import pandas as pd
import plotly.express as px

GRID_PATH = "iowa_grid_lookup.parquet"
PANEL_PATH = "nitrogen_surplus_iowa_grid_panel.parquet"


def plot_iowa_year(
    year,
    value_col="surplus_kgha",
    grid_path=GRID_PATH,
    panel_path=PANEL_PATH,
    downsample=1,
    clip_quantiles=(0.01, 0.99),
):
    """
    Map nitrogen surplus across Iowa's 250m grid for a single year.

    year:            the year to plot (matches the 'year' column in the panel parquet)
    value_col:       column to color by, e.g. "surplus_kgha" or "total_kg_N"
    downsample:      keep every Nth pixel for faster rendering (1 = full resolution,
                      ~2-2.5M points; try 3-5 for smoother interaction)
    clip_quantiles:  clip the color scale to these percentiles so a few extreme
                      pixels don't wash out the color range for everything else
    """
    grid = pd.read_parquet(grid_path)

    # Push the year filter down to the parquet reader so the whole panel
    # (all years) never has to be loaded into memory at once.
    panel = pd.read_parquet(panel_path, filters=[("year", "==", year)])
    if panel.empty:
        raise ValueError(f"No rows found for year={year} in {panel_path}")

    df = panel.merge(grid, on="pixel_id", how="inner")

    if downsample > 1:
        df = df.iloc[::downsample]

    lo, hi = df[value_col].quantile(clip_quantiles)

    fig = px.scatter_mapbox(
        df,
        lat="lat",
        lon="lon",
        color=value_col,
        color_continuous_scale="YlOrRd",
        range_color=(lo, hi),
        mapbox_style="open-street-map",  # free basemap, no API token needed
        zoom=6,
        center={"lat": 42.0, "lon": -93.5},  # roughly the center of Iowa
        height=750,
        opacity=0.8,
        title=f"Nitrogen surplus — Iowa, {year}",
        labels={value_col: "Surplus (kg N/ha)"},
    )
    fig.update_traces(marker=dict(size=4))
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
    return fig


if __name__ == "__main__":
    # Full resolution (downsample=1) is ~2-2.5M points, which can feel sluggish
    # to pan/zoom in a browser. Downsample while exploring, then drop to 1 for
    # a final high-res export.
    fig = plot_iowa_year(2015, downsample=50)
    # fig.show()
    fig.write_html("iowa_surplus_2015_map.html")

In [7]:
import pandas as pd

grid = pd.read_parquet("iowa_grid_lookup.parquet")
panel = pd.read_parquet("nitrogen_surplus_iowa_grid_panel.parquet")

print("grid rows:", len(grid), "| pixel_id dtype:", grid["pixel_id"].dtype)
print("panel rows:", len(panel), "| pixel_id dtype:", panel["pixel_id"].dtype)
print("years available:", sorted(panel["year"].unique()))
print("lat range:", grid["lat"].min(), "to", grid["lat"].max())
print("lon range:", grid["lon"].min(), "to", grid["lon"].max())

year = 2015  # use one of the years printed above
sub = panel[panel["year"] == year]
print("rows for that year:", len(sub))

merged = sub.merge(grid, on="pixel_id", how="inner")
print("rows after merge:", len(merged))
print(merged.sample(10))

grid rows: 2996074 | pixel_id dtype: int32
panel rows: 263654512 | pixel_id dtype: int32
years available: [np.int16(1930), np.int16(1931), np.int16(1932), np.int16(1933), np.int16(1934), np.int16(1935), np.int16(1936), np.int16(1937), np.int16(1938), np.int16(1939), np.int16(1940), np.int16(1941), np.int16(1942), np.int16(1943), np.int16(1944), np.int16(1945), np.int16(1946), np.int16(1947), np.int16(1948), np.int16(1949), np.int16(1950), np.int16(1951), np.int16(1952), np.int16(1953), np.int16(1954), np.int16(1955), np.int16(1956), np.int16(1957), np.int16(1958), np.int16(1959), np.int16(1960), np.int16(1961), np.int16(1962), np.int16(1963), np.int16(1964), np.int16(1965), np.int16(1966), np.int16(1967), np.int16(1968), np.int16(1969), np.int16(1970), np.int16(1971), np.int16(1972), np.int16(1973), np.int16(1974), np.int16(1975), np.int16(1976), np.int16(1977), np.int16(1978), np.int16(1979), np.int16(1980), np.int16(1981), np.int16(1982), np.int16(1983), np.int16(1984), np.int16(1985

In [6]:
merged.describe()

,pixel_id,year,surplus_kgha,total_kg_N,x,y,lon,lat
count,2.996074e+06,2996074.0,2.996074e+06,2.996074e+06,2.996074e+06,2996074.000,2.996074e+06,2.996074e+06
mean,1.498036e+06,2015.0,6.371097e+01,3.981935e+02,2.148750e+05,2113250.000,-9.338850e+01,4.198138e+01
std,8.648922e+05,0.0,5.792525e+01,3.620328e+02,1.542247e+05,101180.625,1.874184e+00,9.049295e-01
min,0.000000e+00,2015.0,0.000000e+00,0.000000e+00,-5.212500e+04,1938125.000,-9.664832e+01,4.032695e+01
25%,7.490182e+05,2015.0,3.620000e+00,2.262500e+01,8.137500e+04,2025625.000,-9.501131e+01,4.119835e+01
50%,1.498036e+06,2015.0,6.634000e+01,4.146250e+02,2.148750e+05,2113250.000,-9.338793e+01,4.198093e+01
75%,2.247055e+06,2015.0,1.034700e+02,6.466875e+02,3.483750e+05,2200875.000,-9.176605e+01,4.276414e+01
max,2.996073e+06,2015.0,3.886100e+02,2.428812e+03,4.818750e+05,2288375.000,-9.001437e+01,4.358816e+01
